# Chapter 5: Data Preprocessing & Feature Engineering (The Data Pipeline)

Raw data is messy. This notebook shows you how to clean, transform, and engineer features before feeding data to a model.

We'll cover:
1. **Why Raw Data Fails** — The Garbage In Problem
2. **Feature Scaling** — StandardScaler vs MinMaxScaler
3. **Handling Missing Data** — Drop vs Impute
4. **Encoding Categorical Data** — LabelEncoder vs OneHotEncoder
5. **Feature Engineering** — Creating derived features
6. **The Pipeline** — Chaining it all together

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 11

---
## 1. Why Raw Data Fails

Let's see what happens when we feed raw, unscaled data to KNN.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

wine = load_wine()
X, y = wine.data, wine.target

print(f"Feature ranges (first 5 features):")
for i in range(5):
    print(f"  {wine.feature_names[i]:>25s}: min={X[:, i].min():.2f}, max={X[:, i].max():.2f}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print(f"\nKNN accuracy (no scaling): {accuracy_score(y_test, knn.predict(X_test)):.2%}")
print("\nThe features have wildly different ranges. KNN uses distance, so large features dominate.")

---
## 2. Feature Scaling

Two approaches: StandardScaler (mean=0, std=1) and MinMaxScaler (range 0-1).

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# StandardScaler
scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_test_std = scaler_std.transform(X_test)

knn_std = KNeighborsClassifier(n_neighbors=5)
knn_std.fit(X_train_std, y_train)
acc_std = accuracy_score(y_test, knn_std.predict(X_test_std))

# MinMaxScaler
scaler_mm = MinMaxScaler()
X_train_mm = scaler_mm.fit_transform(X_train)
X_test_mm = scaler_mm.transform(X_test)

knn_mm = KNeighborsClassifier(n_neighbors=5)
knn_mm.fit(X_train_mm, y_train)
acc_mm = accuracy_score(y_test, knn_mm.predict(X_test_mm))

print(f"KNN accuracy (no scaling):    {accuracy_score(y_test, knn.predict(X_test)):.2%}")
print(f"KNN accuracy (StandardScaler): {acc_std:.2%}")
print(f"KNN accuracy (MinMaxScaler):   {acc_mm:.2%}")
print("\nScaling makes a huge difference for distance-based models!")

In [ ]:
# Visualize the distributions before and after scaling
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
features_to_show = [0, 1, 2, 3]
names = [wine.feature_names[i] for i in features_to_show]

for ax, data, title in zip(axes, [X_train[:, features_to_show], X_train_std[:, features_to_show], X_train_mm[:, features_to_show]],
                            ['Original', 'StandardScaler', 'MinMaxScaler']):
    bp = ax.boxplot([data[:, i] for i in range(4)], labels=names, patch_artist=True, widths=0.5)
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=25)

plt.suptitle('Feature Distributions: Before and After Scaling', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Handling Missing Data

Real data has holes. Let's inject missing values and compare strategies.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

# Create clean dataset
X_clean, y_clean = make_classification(n_samples=500, n_features=10, n_informative=5,
                                        random_state=42, n_classes=3)
X_tr, X_te, y_tr, y_te = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

# Baseline
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
print(f"Baseline (no missing data): {accuracy_score(y_te, rf.predict(X_te)):.2%}")

# Inject 30% missing values
rng = np.random.RandomState(42)
X_tr_miss = X_tr.copy()
X_te_miss = X_te.copy()
X_tr_miss[rng.random(X_tr_miss.shape) < 0.3] = np.nan
X_te_miss[rng.random(X_te_miss.shape) < 0.3] = np.nan

print(f"\nMissing values injected: ~30% of all values")
print(f"Training rows with at least one NaN: {np.sum(np.any(np.isnan(X_tr_miss), axis=1))}/{len(X_tr_miss)}")

In [ ]:
# Strategy 1: Drop rows
valid_tr = ~np.any(np.isnan(X_tr_miss), axis=1)
valid_te = ~np.any(np.isnan(X_te_miss), axis=1)
rf_drop = RandomForestClassifier(n_estimators=100, random_state=42)
rf_drop.fit(X_tr_miss[valid_tr], y_tr[valid_tr])
acc_drop = accuracy_score(y_te[valid_te], rf_drop.predict(X_te_miss[valid_te]))
print(f"Drop rows:      {acc_drop:.2%} (kept {np.sum(valid_tr)}/{len(X_tr_miss)} training rows)")

# Strategy 2: Mean imputation
imp_mean = SimpleImputer(strategy='mean')
X_tr_mean = imp_mean.fit_transform(X_tr_miss)
X_te_mean = imp_mean.transform(X_te_miss)
rf_mean = RandomForestClassifier(n_estimators=100, random_state=42)
rf_mean.fit(X_tr_mean, y_tr)
acc_mean = accuracy_score(y_te, rf_mean.predict(X_te_mean))
print(f"Impute (mean):  {acc_mean:.2%} (kept all rows)")

# Strategy 3: Median imputation
imp_med = SimpleImputer(strategy='median')
X_tr_med = imp_med.fit_transform(X_tr_miss)
X_te_med = imp_med.transform(X_te_miss)
rf_med = RandomForestClassifier(n_estimators=100, random_state=42)
rf_med.fit(X_tr_med, y_tr)
acc_med = accuracy_score(y_te, rf_med.predict(X_te_med))
print(f"Impute (median): {acc_med:.2%} (kept all rows)")

print("\nImputation preserves data. Dropping rows wastes it.")

---
## 4. Encoding Categorical Data

Models need numbers. Strings like 'red' and 'blue' must be converted.

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

colors = ['red', 'blue', 'green', 'red', 'yellow', 'blue']

# LabelEncoder: each category -> integer
le = LabelEncoder()
le_result = le.fit_transform(colors)
print("LabelEncoder:")
for c, v in zip(colors, le_result):
    print(f"  {c} -> {v}")

print(f"\nProblem: model thinks yellow ({le_result[-2]}) > green ({le_result[2]}) > blue ({le_result[1]})")

# OneHotEncoder: each category -> binary vector
ohe = OneHotEncoder(sparse_output=False)
ohe_result = ohe.fit_transform(np.array(colors).reshape(-1, 1))
print(f"\nOneHotEncoder:")
print(f"Categories: {ohe.categories_[0]}")
for c, v in zip(colors, ohe_result):
    print(f"  {c} -> {v.astype(int)}")

print("\nNo false ordering. Each category is independent.")

---
## 5. Feature Engineering

Sometimes the best feature isn't in your raw data. You have to create it.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Create data where the target depends on area (length * width)
np.random.seed(42)
n = 500
length = np.random.uniform(1, 10, n)
width = np.random.uniform(1, 10, n)
noise = np.random.normal(0, 0.5, n)
y_eng = (length * width + noise > 25).astype(int)

# Raw features only
X_raw = np.column_stack([length, width])
X_tr_r, X_te_r, y_tr_e, y_te_e = train_test_split(X_raw, y_eng, test_size=0.2, random_state=42)

dt_raw = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_raw.fit(X_tr_r, y_tr_e)
print(f"Decision Tree (length, width only):        {accuracy_score(y_te_e, dt_raw.predict(X_te_r)):.2%}")

# Add engineered feature: area
X_eng = np.column_stack([length, width, length * width])
X_tr_e, X_te_e_full = train_test_split(X_eng, y_eng, test_size=0.2, random_state=42)[:2]

dt_eng = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_eng.fit(X_tr_e, y_tr_e)
print(f"Decision Tree (+ area = length * width):   {accuracy_score(y_te_e, dt_eng.predict(X_te_e_full)):.2%}")

print("\nThe derived feature captures the actual relationship in the data.")

---
## 6. The Pipeline

Chain preprocessing + model into a single object. Like middleware in Express.js.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

wine = load_wine()
X_w, y_w = wine.data, wine.target

# Without pipeline
knn_no_pipe = KNeighborsClassifier(n_neighbors=5)
scores_no_pipe = cross_val_score(knn_no_pipe, X_w, y_w, cv=5, scoring='accuracy')
print(f"KNN (no pipeline, no scaling): {scores_no_pipe.mean():.3f} +/- {scores_no_pipe.std():.3f}")

# With pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])
scores_pipe = cross_val_score(pipe, X_w, y_w, cv=5, scoring='accuracy')
print(f"KNN (pipeline with scaling):   {scores_pipe.mean():.3f} +/- {scores_pipe.std():.3f}")

print("\nThe pipeline handles fit/transform automatically. No data leakage.")
print("One object to save, load, and deploy.")

In [ ]:
# You can inspect pipeline steps
print("Pipeline steps:")
for name, step in pipe.steps:
    print(f"  {name}: {step.__class__.__name__}")

# You can access individual steps
print(f"\nScaler type: {pipe.named_steps['scaler']}")
print(f"Classifier type: {pipe.named_steps['classifier']}")

# Swap the classifier — the pipeline interface stays the same
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
scores_rf = cross_val_score(pipe_rf, X_w, y_w, cv=5, scoring='accuracy')
print(f"\nRandom Forest (pipeline with scaling): {scores_rf.mean():.3f} +/- {scores_rf.std():.3f}")

---
## Challenge

1. Build a `Pipeline` with `StandardScaler` + `KNeighborsClassifier` on the digits dataset. Compare cross-validation scores with and without scaling.
2. Try `MinMaxScaler` instead of `StandardScaler`. Is there a difference?
3. Does scaling help `RandomForestClassifier`? Why or why not?

**Next Chapter**: We've been using default settings for everything. Next, we'll explore **Hyperparameter Tuning** — finding the best configuration for your model.